In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import pickle 
import tifffile as tf
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
from scipy.spatial import cKDTree
from scipy.stats import wasserstein_distance, sem

In [ ]:
binary_mask_path = Path(r"") # path to binary mask tissue = 1 outside tissue = 0
segmentation_mask_path = Path(r"") # path to cell segmentation masks 

In [ ]:
tissue_mask_paths = [f for f in binary_mask_path.rglob('*.tif') if 'mask' in str(f)]
cell_segmentation_mask_paths = [f for f in segmentation_mask_path.rglob('*.pickle') if 'mask' in str(f)]

In [ ]:
tissue_masks = []

In [ ]:
cycles = []

In [ ]:
for path in tissue_mask_paths: 
    tissue_masks.append(tf.imread(path))
    cycle_index = str(path).find('Cycle')
    cycle = str(path)[cycle_index: cycle_index + len('Cycle')+2]
    cycles.append(cycle)

In [ ]:
segmentation_masks = []

In [ ]:
for i, path in tqdm(enumerate(cell_segmentation_mask_paths)):
    with open(path, 'rb') as f: 
        segmentation_mask = pickle.load(f)
    segmentation_mask[tissue_masks[i] == 0] = 0
    segmentation_masks.append(segmentation_mask)

In [ ]:

def calculate_normal_displacement(mask_cycle_1, mask_cycle_2):
    mask_cycle_1 = mask_cycle_1.astype(bool)
    mask_cycle_2 = mask_cycle_2.astype(bool)
    boundary_cycle_1 = mask_cycle_1 ^ binary_erosion(mask_cycle_1)
    boundary_cycle_2 = mask_cycle_2 ^ binary_erosion(mask_cycle_2)

    distance_to_cycle_2 = distance_transform_edt(~boundary_cycle_2)

    signed_distance = distance_to_cycle_2.copy()
    signed_distance[~mask_cycle_2] *= -1

    gradient_y, gradient_x = np.gradient(mask_cycle_1.astype(float))

    normal_length = np.sqrt(
        gradient_x**2 + gradient_y**2
    )

    normal_length[normal_length == 0] = 1

    displacement = np.full(mask_cycle_1.shape, np.nan)
    displacement[boundary_cycle_1] = signed_distance[boundary_cycle_1]

    return displacement



In [ ]:

def compare_tissues(
    cells_1,
    cells_2,
    tissue_1,
    tissue_2,
    pixel_size,
    normalize_by_tissue_area,
):

    def spacing(mask, tissue):
        mask = np.asarray(mask)
        _, inverse, counts = np.unique(
            mask[mask > 0], return_inverse=True, return_counts=True
        )

        y, x = np.nonzero(mask > 0)
        cy = np.bincount(inverse, weights=y) / counts
        cx = np.bincount(inverse, weights=x) / counts

        keep = counts >= 0

        if tissue is not None:
            tissue = np.asarray(tissue, dtype=bool)
            iy = np.clip(np.rint(cy).astype(int), 0, tissue.shape[0] - 1)
            ix = np.clip(np.rint(cx).astype(int), 0, tissue.shape[1] - 1)
            keep &= tissue[iy, ix]

        coords = np.column_stack((cy[keep], cx[keep])) * pixel_size

        return cKDTree(coords).query(coords, k=2, workers=-1)[0][:, 1]

    distances_1 = spacing(cells_1, tissue_1)
    distances_2 = spacing(cells_2, tissue_2)

    if normalize_by_tissue_area:
        area_1 = np.count_nonzero(tissue_1) * pixel_size**2
        area_2 = np.count_nonzero(tissue_2) * pixel_size**2

        distances_1 /= np.sqrt(area_1)
        distances_2 /= np.sqrt(area_2)

    return wasserstein_distance(
            distances_1, distances_2
        )
    

In [ ]:
boundary_displacement = {}

In [ ]:
scale = 0.1771*4

In [ ]:
for i in tqdm(range(len(cycles[:-1]))):
    displacement = calculate_normal_displacement(tissue_masks[i], tissue_masks[i+1])
    displacement = displacement[~np.isnan(displacement)] 
    displacement_um = [d*scale for d in displacement]
    boundary_displacement[cycles[i] + '/' + cycles[i+1]] = displacement_um

In [ ]:
cell_density_distribution_distance = {}

In [ ]:
for i in tqdm(range(len(cycles[:-1]))):
    cell_density_distribution_distance[cycles[i] + '/' + cycles[i+1]] = compare_tissues(segmentation_masks[i], segmentation_masks[i+1], tissue_masks[i], tissue_masks[i+1], scale, False)

In [ ]:
def remove_second_occurrence(text, sub):
    parts = text.split(sub)
    if len(parts) > 2:
        return sub.join(parts[:2]) + parts[2]
    return text

In [ ]:
labels = []
for cycle_pair_key in list(boundary_displacement.keys()):
    labels.append(remove_second_occurrence(cycle_pair_key, 'Cycle '))

data = list(boundary_displacement.values())

means = [np.nanmean(np.asarray(d, dtype=float)) for d in data]
standard_errors = [
    sem(np.asarray(d, dtype=float), nan_policy="omit")
    for d in data
]

plt.rcParams.update({'font.size': 12})
plt.figure(figsize=(4, 4))

plt.bar(
    labels,
    means,
    width=0.5,
    yerr=standard_errors,
    capsize=5,
    color="steelblue",
    edgecolor="black",
    error_kw={"elinewidth": 1.2}
)

plt.axhline(0, color="black", linewidth=1, linestyle="--")

plt.ylabel("Boundary displacement ($\mu$m)")
plt.xlabel("Cycle pair")

plt.xticks(rotation=45, ha="right")
plt.yticks(plt.yticks()[0][::2])

plt.tight_layout()
plt.show()

In [ ]:

data = list(cell_density_distribution_distance.values())

means = [np.nanmean(np.asarray(d, dtype=float)) for d in data]
standard_errors = [
    sem(np.asarray(d, dtype=float), nan_policy="omit")
    for d in data
]

plt.rcParams.update({'font.size': 12})
plt.figure(figsize=(4, 4))

plt.bar(
    labels,
    means,
    width=0.5,
    yerr=standard_errors,
    capsize=5,
    color="steelblue",
    edgecolor="black",
    error_kw={"elinewidth": 1.2}
)

plt.axhline(0, color="black", linewidth=1, linestyle="--")

plt.ylabel("Wasserstein distance")
plt.xlabel("Cycle pair")

plt.xticks(rotation=45, ha="right")
plt.ylim([0, 1])

plt.tight_layout()
plt.show()